<a href="https://colab.research.google.com/github/dinaglamshowroom/projet-data_oc/blob/main/Projet_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Projet 6 Openclassroom - Détectez des faux billets

# Introduction & Objectifs

## Contexte du projet

## Objectif principal de l’étude

Construire un modèle explicatif du revenu futur d’un individu, en se basant sur des données très limitées :

* le revenu des parents,

* le revenu moyen de son pays,

* l’indice d’inégalité (indice de Gini) de ce pays.

L’enjeu est de prédire la capacité future de revenu d’un jeune prospect, enfant de client bancaire actuel, en vue de le cibler commercialement comme client à fort potentiel.

**Mission 1 – Exploration des données**


* Décrire le périmètre des données (année, couverture mondiale, nombre de pays, etc.)

* Identifier les quantiles utilisés (ici, des centiles)

* Justifier l’usage de ces quantiles : représentativité, granularité

* Expliquer la pertinence du $PPP pour la comparaison entre pays

**Mission 2 – Visualisations & inégalités**


* Visualiser la diversité des distributions de revenu (log(income) vs quantiles)

* Tracer la courbe de Lorenz pour plusieurs pays (visualisation de l’inégalité)

* Visualiser l’évolution de l’indice de Gini au fil des ans

* Comparer les pays : classement par Gini (top 5 égalitaires/inégalitaires + position de la France)

**Mission 3 – Simulation des revenus parentaux**


* Recréer une classe de revenu parentale pour chaque individu

* Utiliser la formule : (ln(Ychlid) = a + pj * ln(Yparents) + e

* Générer de grandes populations simulées pour estimer : P(Cparent|Cchlidpj)


* Produire un nouvel échantillon enrichi où chaque individu a :

  * un pays

  * un revenu moyen du pays

  * un indice de Gini

  * une classe de revenu parentale (simulée)

**Mission 4 – Modélisation statistique**



* ANOVA : mesurer la part de variance expliquée uniquement par le pays

* Régression linéaire 1 :

  * Variables explicatives : revenu moyen du pays + Gini

  * Calcul de la variance expliquée

  * Interprétation : que reste-t-il comme variance non expliquée ? (efforts, chance…)

* Régression linéaire 2 :

  * On ajoute la classe de revenu des parents

  * Nouvelle variance expliquée

  * Interprétation du coefficient Gini : favorise-t-il les riches ?

* Décomposition finale de la variance :

  * Part expliquée par le pays et la classe sociale d’origine

  * Part inexpliquée : mobilité sociale, aléa, mérite, etc.

## Notions à connaitre:

1. **Parité de Pouvoir d’Achat (PPP)**

L’unité des revenus dans les données est le PPP dollar, qui permet de comparer les revenus à travers les pays en neutralisant les différences de niveau de vie.

Ex : 1$ PPP représente le même panier de biens aux USA, en Inde ou en France.

2. **Indice de Gini**

Mesure l’inégalité des revenus dans un pays.

0 = parfaite égalité

1 = inégalité totale (une seule personne a tous les revenus)

3. **Quantiles**

Les données de revenu sont organisées en centiles (100 classes). Chaque individu est représenté par la classe de revenu à laquelle il appartient dans son pays.

4. **Élasticité intergénérationnelle (ρj)**

C’est un coefficient de transmission du revenu des parents à l’enfant.
Plus il est élevé, plus le revenu des parents détermine fortement celui de l’enfant (faible mobilité sociale).
C’est un paramètre crucial pour simuler les revenus des parents à partir des données disponibles.

** Résumé des Étapes Techniques**

* Importer et nettoyer les données (WID + Gini + population)

* Analyser les quantiles de revenus pour chaque pays

* Tracer les courbes : revenu/log-revenu vs quantiles

* Calculer ou importer les élasticités intergénérationnelles

* Simuler les classes parentales à partir des distributions

* Créer un super-échantillon 500x plus grand

* Construire les modèles statistiques

  * ANOVA

  * Régressions linéaires

* Comparer les performances des modèles

  * via le R²

  * via l’analyse des résidus (part inexpliquée)

## Question

1) **Générez un grand nombre n**
 de réalisations d'une variable que nous appellerons  ln(Yparent)
 selon une loi normale. Le choix de la moyenne et de l'écart type n'auront pas d'incidence sur le résultat final. n
 doit être supérieur à 1000 fois le nombre de quantiles.


2) **Générez n**
 réalisations du terme d'erreur ϵ
 selon une loi normale de moyenne 0 et d'écart type 1.


3) Pour une valeur donnée de ρj (par exemple 0.9), calculez ychild=eα+ρjln(yparent)+ϵ
 . Le choix de α
 n'a aucune incidence sur le résultat final et peut être supprimé. À ce stade, ychild
 contient des valeurs dont l'ordre de grandeur ne reflète pas la réalité, mais cela n3'a pas d'influence pour la suite.


4) Pour chacun des n
 individus générés, calculez la classe de revenu  ci,child
 ainsi que la classe de revenu de ses parents ci,parent
 , à partir de ychild
 et yparent
.


5) À partir de cette dernière information, estimez pour chaque  ci,child
 la distribution conditionnelle de ci,parent
 . Par exemple, si vous observez 6 individus ayant à la fois  ci,child=5
 et ci,parent=8
 , et que 200 individus sur 20000 ont ci,child=5
 , alors la probabilité d'avoir ci,parent=8
 sachant  ci,child=5
 et sachant ρj=0.9
 sera estimée à 6/200 (On note cette probabilité comme ceci : P(ci,parent=8|ci,child=5,ρj=0.9)=0.03). Si votre population est divisée en c
 classes de revenu, vous devriez alors avoir c2
estimations de ces probabilités conditionnelles, pour chaque pays.


6) Optionnellement et pour vérifier la cohérence de votre code, vous pouvez créer un graphique représentant ces distributions conditionnelles. Voici 2 exemples pour une population segmentée en 10 classes, pour 2 valeurs de ρj
 : l'une traduisant une forte mobilité (0.1) et l'autre une très faible mobilité (0.9) :

# Chargement & Préparation des Données

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import os, glob
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy import stats
from sklearn import datasets
import zipfile
import os
from google.colab import files


In [5]:
# Création du dossier de stockage
os.makedirs("plots", exist_ok=True)

# Compteur de plots
_plot_counter = 1

# On intercepte matplotlib.show()
_old_show = plt.show

def _new_show(*args, **kwargs):
    global _plot_counter
    nom = f"plots/plot_{_plot_counter:03d}.png"
    plt.savefig(nom, dpi=300, bbox_inches="tight")
    _plot_counter += 1
    _old_show(*args, **kwargs)

plt.show = _new_show

## Importation du dataset

In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
DATA_DIR = "/content/drive/MyDrive/Projet_7"
csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))

print("Fichiers CSV trouvés :")
for f in csv_files:
    print(f)

Fichiers CSV trouvés :
/content/drive/MyDrive/Projet_7/data-projet7.csv
/content/drive/MyDrive/Projet_7/GDIMMay2018+(1).csv
/content/drive/MyDrive/Projet_7/world_income_distribution.csv
/content/drive/MyDrive/Projet_7/GINI.csv
/content/drive/MyDrive/Projet_7/GINI2.csv


In [8]:
dfs = {f.split("/")[-1]: pd.read_csv(f) for f in csv_files}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")

data-projet7.csv: (11599, 6)
GDIMMay2018+(1).csv: (6504, 66)
world_income_distribution.csv: (11599, 7)
GINI.csv: (271, 11)
GINI2.csv: (1897, 4)


In [9]:
df_income = dfs["world_income_distribution.csv"]
df_gini = dfs["GINI.csv"]
df_data7 = dfs["data-projet7.csv"]
df_GDI = dfs["GDIMMay2018+(1).csv"]

## Description des variables

In [10]:
df_income.describe()

,year_survey,quantile,nb_quantiles,income,pop,gdpppp
count,11599.000000,11599.000000,11599.0,11599.000000,11599.000000,1.139900e+04
mean,2007.982757,50.500819,100.0,6069.224260,16.178084,5.022128e+04
std,0.909633,28.868424,0.0,9414.185972,134.619280,4.000688e+05
min,2004.000000,1.000000,100.0,16.719418,0.003100,3.031931e+02
25%,2008.000000,25.500000,100.0,900.685515,0.052900,2.576000e+03
50%,2008.000000,51.000000,100.0,2403.244900,0.146998,7.560000e+03
75%,2008.000000,75.500000,100.0,7515.420900,0.456360,1.877300e+04
max,2011.000000,100.000000,100.0,176928.550000,1418.000000,4.300332e+06


## Vérification des données

### df_GDI

In [11]:
df_GDI.head(5)

,countryname,wbcode,iso3,region,incgroup2,incgroup4,fragile,survey,year,status,...,Cores2125_MAcatC1,Shortfall0611_obs,Shortfall0611_IGP,Shortfall1217_obs,Shortfall1217_IGP,IGEincome,S1,S2,S3,MLD_psu
0,Afghanistan,AFG,AFG,South Asia,Developing economies,Low income,1,NRVA,1980,Co-residents only,...,NaN,25103.0,0.086197,18054.0,0.345224,NaN,NaN,NaN,NaN,0.1
1,Afghanistan,AFG,AFG,South Asia,Developing economies,Low income,1,NRVA,1980,Co-residents only,...,NaN,12107.0,0.083271,8538.0,0.389952,NaN,NaN,NaN,NaN,0.1
2,Afghanistan,AFG,AFG,South Asia,Developing economies,Low income,1,NRVA,1980,Co-residents only,...,NaN,12996.0,0.089161,9516.0,0.307687,NaN,NaN,NaN,NaN,0.1
3,Afghanistan,AFG,AFG,South Asia,Developing economies,Low income,1,NRVA,1980,Co-residents only,...,NaN,25396.0,0.050447,18387.0,0.218062,NaN,NaN,NaN,NaN,0.1
4,Afghanistan,AFG,AFG,South Asia,Developing economies,Low income,1,NRVA,1980,Co-residents only,...,NaN,12246.0,0.047961,8677.0,0.230909,NaN,NaN,NaN,NaN,0.1


In [12]:
df_GDI.describe()

,fragile,year,cohort,obs,P1,P2,P3,P4,P5,C1,...,Cores2125_MAcatC1,Shortfall0611_obs,Shortfall0611_IGP,Shortfall1217_obs,Shortfall1217_IGP,IGEincome,S1,S2,S3,MLD_psu
count,6504.000000,6504.000000,6504.000000,6501.000000,6501.000000,6501.000000,6501.000000,6501.000000,6501.000000,6501.000000,...,845.000000,930.000000,908.000000,1361.00000,1337.000000,853.000000,2845.000000,2845.000000,2845.000000,1225.000000
mean,0.089483,1961.615929,1961.615929,1286.105830,0.361102,0.206518,0.149553,0.188190,0.094407,0.219147,...,0.612634,3395.493548,0.079916,2134.59662,0.172016,0.516928,0.107122,0.053301,0.839577,0.251861
std,0.285462,14.412033,14.412033,2291.970266,0.345911,0.167120,0.130607,0.184797,0.114704,0.289676,...,0.180464,3995.790251,0.103659,3265.66739,0.139615,0.249615,0.061527,0.034521,0.074349,0.124587
min,0.000000,1940.000000,1940.000000,50.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.128471,1.000000,-0.593894,1.00000,-0.130202,0.112876,0.009765,0.012044,0.649935,0.030000
25%,0.000000,1950.000000,1950.000000,283.000000,0.032625,0.065574,0.042012,0.032678,0.013502,0.002172,...,0.492818,1058.750000,0.028097,159.00000,0.056494,0.312578,0.057830,0.027947,0.782551,0.160000
50%,0.000000,1960.000000,1960.000000,514.000000,0.228595,0.171569,0.116383,0.121529,0.049856,0.040877,...,0.633241,2167.000000,0.069259,1159.00000,0.158314,0.464077,0.098471,0.048137,0.855454,0.210000
75%,0.000000,1980.000000,1980.000000,1242.000000,0.693064,0.311515,0.228381,0.312678,0.135076,0.397490,...,0.756912,4060.750000,0.107568,2458.00000,0.264241,0.689613,0.152865,0.071971,0.899058,0.320000
max,1.000000,1980.000000,1980.000000,29657.000000,1.000000,0.845898,0.777907,0.834078,0.789995,1.000000,...,0.962139,29106.000000,1.247346,25368.00000,0.830648,1.095440,0.292185,0.175662,0.960701,0.680000


In [13]:
df_GDI.duplicated().sum()

np.int64(0)

In [14]:
nb_pays = df_GDI["countryname"].nunique()
print(f"Nombre de pays distincts : {nb_pays}")

Nombre de pays distincts : 150


### df_income

In [15]:

df_income.head(5)

,country,year_survey,quantile,nb_quantiles,income,pop,gdpppp
0,ALB,2008,1,100,728.89795,0.03143,7297.0
1,ALB,2008,2,100,916.66235,0.03143,7297.0
2,ALB,2008,3,100,1010.91600,0.03143,7297.0
3,ALB,2008,4,100,1086.90780,0.03143,7297.0
4,ALB,2008,5,100,1132.69970,0.03143,7297.0


In [16]:
df_income.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11599 entries, 0 to 11598
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country       11599 non-null  object 
 1   year_survey   11599 non-null  int64  
 2   quantile      11599 non-null  int64  
 3   nb_quantiles  11599 non-null  int64  
 4   income        11599 non-null  float64
 5   pop           11599 non-null  float64
 6   gdpppp        11399 non-null  float64
dtypes: float64(3), int64(3), object(1)
memory usage: 634.4+ KB


In [17]:
annees = df_income["year_survey"].unique()
print(f"Année(s) des enquêtes : {annees}")

Année(s) des enquêtes : [2008 2009 2010 2007 2006 2011 2004]


In [18]:
for year in annees:
    df_year = df_income[df_income['year_survey'] == year]
    print(f"Année {year} : {df_year.shape[0]} éléments")

Année 2008 : 7599 éléments
Année 2009 : 1200 éléments
Année 2010 : 600 éléments
Année 2007 : 1500 éléments
Année 2006 : 500 éléments
Année 2011 : 100 éléments
Année 2004 : 100 éléments


Les enquêtes de revenu utilisées dans ce projet s’étendent sur plusieurs années, principalement autour de 2008, mais incluent également des données de 2004 à 2011.

In [19]:
n_pays = df_income["country"].nunique()
print(f"Nombre de pays distincts : {n_pays}")


Nombre de pays distincts : 116


In [20]:
df_income['country'].unique()


array(['ALB', 'ARG', 'ARM', 'AUT', 'AZE', 'BEL', 'BFA', 'BGD', 'BGR',
       'BIH', 'BLR', 'BOL', 'BRA', 'BTN', 'CAF', 'CAN', 'CHL', 'CHN',
       'CIV', 'CMR', 'COL', 'CRI', 'CYP', 'CZE', 'DEU', 'DNK', 'DOM',
       'ECU', 'EGY', 'ESP', 'EST', 'FIN', 'FJI', 'FRA', 'GBR', 'GEO',
       'GHA', 'GIN', 'GRC', 'GTM', 'HND', 'HRV', 'HUN', 'IDN', 'IND',
       'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JOR', 'JPN', 'KAZ',
       'KEN', 'KGZ', 'KHM', 'KOR', 'XKX', 'LAO', 'LBR', 'LKA', 'LTU',
       'LUX', 'LVA', 'MAR', 'MDA', 'MDG', 'MEX', 'MKD', 'MLI', 'MNE',
       'MNG', 'MOZ', 'MRT', 'MWI', 'MYS', 'NER', 'NGA', 'NIC', 'NLD',
       'NOR', 'NPL', 'PAK', 'PAN', 'PER', 'PHL', 'POL', 'PRT', 'PRY',
       'ROU', 'RUS', 'SDN', 'SLV', 'SRB', 'SVK', 'SVN', 'SWE', 'SWZ',
       'SYR', 'THA', 'TJK', 'TLS', 'TUR', 'TWN', 'TZA', 'UGA', 'UKR',
       'URY', 'USA', 'VEN', 'VNM', 'PSE', 'YEM', 'ZAF', 'COD'],
      dtype=object)

In [21]:
df_income.duplicated().sum()

np.int64(0)

In [22]:
df_income.nb_quantiles.nunique()


1

In [23]:
df_income.isnull().sum()

,0
country,0
year_survey,0
quantile,0
nb_quantiles,0
income,0
pop,0
gdpppp,200


In [24]:
# on cherche les pays correspondant aux NaN
print(df_income[df_income['gdpppp'].isna()].country.unique())

['XKX' 'PSE']


on remarque que les valeurs **gdpppp** sont manquantes  pour les codes pays **XKX (Kosovo)** et **PSE (Occupied Palestinian Territory)**

Nous allons chercher  les valeurs manquantes sur le site de "word Bank Group" et et regarder le **WDI** (NY.GDP.PCAP.PP.CD):

* **Kosovo (XKX)**: gdpppp 2008 = **7 249** $

* **Territoires palestiniens** (PSE): gdpppp 2008 = **4 972** $

In [25]:
df_income.loc[df_income.country=='XKX','gdpppp'] = 7249
df_income.loc[df_income.country=='PSE','gdpppp'] = 4972

In [26]:
df_income.loc[df_income.country=='XKX']

,country,year_survey,quantile,nb_quantiles,income,pop,gdpppp
5800,XKX,2008,1,100,437.89370,0.02,7249.0
5801,XKX,2008,2,100,508.17133,0.02,7249.0
5802,XKX,2008,3,100,591.82820,0.02,7249.0
5803,XKX,2008,4,100,668.00000,0.02,7249.0
5804,XKX,2008,5,100,730.40220,0.02,7249.0
...,...,...,...,...,...,...,...
5895,XKX,2008,96,100,5155.36470,0.02,7249.0
5896,XKX,2008,97,100,5689.52930,0.02,7249.0
5897,XKX,2008,98,100,6233.73930,0.02,7249.0
5898,XKX,2008,99,100,7366.67700,0.02,7249.0


**REMARQUE** La population associée au code XKX est incohérente avec celle du Kosovo seul.
Cela provient du fait qu’en 2008, les séries statistiques WDI ne séparaient pas encore le Kosovo de la Serbie : XKX renvoie donc à  **“Serbie + Kosovo”**.

In [27]:
df_income.loc[df_income.country=='PSE']

,country,year_survey,quantile,nb_quantiles,income,pop,gdpppp
11199,PSE,2009,1,100,195.28990,0.04,4972.0
11200,PSE,2009,2,100,264.36533,0.04,4972.0
11201,PSE,2009,3,100,301.44672,0.04,4972.0
11202,PSE,2009,4,100,329.83392,0.04,4972.0
11203,PSE,2009,5,100,348.76495,0.04,4972.0
...,...,...,...,...,...,...,...
11294,PSE,2009,96,100,2763.88480,0.04,4972.0
11295,PSE,2009,97,100,3077.83330,0.04,4972.0
11296,PSE,2009,98,100,3449.22240,0.04,4972.0
11297,PSE,2009,99,100,4165.99700,0.04,4972.0


On à la même incohérence au niveau de la population, après recherche:


**PSE = Palestine + diaspora** → explique le poids démographique artificiellement élevé.

In [28]:
df_income.isna().sum()

,0
country,0
year_survey,0
quantile,0
nb_quantiles,0
income,0
pop,0
gdpppp,0


In [29]:
df_income["nb_quantiles"].value_counts()


,count
nb_quantiles,
100,11599


Les revenus de chaque pays sont découpés en 100 quantiles

➡  il s’agit donc d’un **échantillonnage en centiles**.

L’usage des centiles permet une granularité maximale dans la représentation de l’échelle des revenus.
Cela facilite une analyse fine des inégalités, en mettant en évidence les écarts entre les couches extrêmes (1er centile vs 100e).
De plus, cet échantillonnage garantit une comparabilité directe entre pays, quel que soit leur niveau de développement.

**Remarque**:

Pour effectuer notre étude on suppose que chaque pays dispose de 100 quantiles:

Or dans df_income: n_pays contient 116 pays pour 11599 observations

➡  **il manque donc un quantile dans les données.**

In [30]:
# Liste des pays incomplets
pays_incomplets = []

# Parcours de tous les pays
for country in df_income["country"].unique():
    quantiles_pays = set(df_income[df_income["country"] == country]["quantile"])
    missing_q = sorted(set(range(1, 101)) - quantiles_pays)
    if missing_q:
        pays_incomplets.append((country, missing_q))

# Affichage des résultats
print(f"Nombre de pays avec données incomplètes : {len(pays_incomplets)}\n")
for pays, manquants in pays_incomplets:
    print(f" {pays} : {len(manquants)} quantile manquant : {manquants}")


Nombre de pays avec données incomplètes : 1

 LTU : 1 quantile manquant : [41]


La distribution des revenus n’est pas complète pour **LTU** : il manque **le 41e centile**, ce qui peut fausser les calculs futurs.

Pour corriger cela on aurait plusieurs options comme:
* Supprimer le pays ➡ as pertinent ici : la Lituanie est presque complète, ce serait du gaspillage.

* Régression locale ou spline ➡ trop lourd pour un seul point.

* On va donc procéder a une **interpolation linéaire**:
  * ➡  utiliser les centiles 40 et 42, et calculer la moyenne pour estimer le 41e.



In [31]:
# Sélection du pays
df_ltu = df_income[df_income["country"] == "LTU"]

# Extraire les lignes des quantiles 40 et 42
row_40 = df_ltu[df_ltu["quantile"] == 40]
row_42 = df_ltu[df_ltu["quantile"] == 42]

# Vérification
if not row_40.empty and not row_42.empty:
    # Fusionner les deux lignes en un seul DataFrame temporaire
    tmp = pd.concat([row_40, row_42])

    # Identifier les colonnes numériques à interpoler
    colonnes_numeriques = tmp.select_dtypes(include=["float64", "int64"]).columns

    # Interpolation linéaire sur ces colonnes
    valeurs_interp = tmp[colonnes_numeriques].mean().to_dict()

    # Création de la nouvelle ligne
    nouvelle_ligne = {
        "country": "LTU",
        "year_survey": int(row_40["year_survey"].values[0]),
        "quantile": 41,
        "nb_quantiles": 100,
        **valeurs_interp
    }

    # Ajouter la ligne au DataFrame principal
    df_income = pd.concat([df_income, pd.DataFrame([nouvelle_ligne])], ignore_index=True)

    # Réordonner le DataFrame
    df_income = df_income.sort_values(by=["country", "quantile"]).reset_index(drop=True)

    print(" Le quantile 41 a été interpolé et ajouté pour la Lituanie.")

else:
    print(" Impossible d’interpoler : quantiles 40 ou 42 manquants.")
print(nouvelle_ligne)



 Le quantile 41 a été interpolé et ajouté pour la Lituanie.
{'country': 'LTU', 'year_survey': 2008.0, 'quantile': 41.0, 'nb_quantiles': 100.0, 'income': 4882.14065, 'pop': 0.0335999987999999, 'gdpppp': 17571.0}


In [32]:
# @title
import pandas as pd



# Vérification des quantiles manquants pour LTU
df_ltu = df_income[df_income["country"] == "LTU"]
quantiles_ltu = df_ltu["quantile"].tolist()
if 41 in quantiles_ltu:
    print("✅ Le quantile 41 est déjà présent pour LTU.")
else:
    # Récupération des lignes 40 et 42
    row_40 = df_ltu[df_ltu["quantile"] == 40]
    row_42 = df_ltu[df_ltu["quantile"] == 42]

    if not row_40.empty and not row_42.empty:
        # Colonnes à interpoler
        income = (row_40["income"].values[0] + row_42["income"].values[0]) / 2
        pop = (row_40["pop"].values[0] + row_42["pop"].values[0]) / 2
        gdpppp = (row_40["gdpppp"].values[0] + row_42["gdpppp"].values[0]) / 2
        year = int(row_40["year_survey"].values[0])

        # Création de la ligne manquante
        ligne_41 = {
            "country": "LTU",
            "year_survey": year,
            "quantile": 41,
            "nb_quantiles": 100,
            "income": income,
            "pop": pop,
            "gdpppp": gdpppp
        }

        # Ajout au DataFrame
        df_income = pd.concat([df_income, pd.DataFrame([ligne_41])], ignore_index=True)

        # Tri final
        df_income = df_income.sort_values(by=["country", "quantile"]).reset_index(drop=True)

        print("✅ Le quantile 41 a été ajouté avec succès pour LTU.")
    else:
        print("❌ Les quantiles 40 ou 42 sont manquants. Interpolation impossible.")


✅ Le quantile 41 est déjà présent pour LTU.


In [33]:
df_income[(df_income["country"] == "LTU") & (df_income["quantile"].between(38, 45))]


,country,year_survey,quantile,nb_quantiles,income,pop,gdpppp
6237,LTU,2008.0,38.0,100.0,4756.43360,0.0336,17571.0
6238,LTU,2008.0,39.0,100.0,4802.36800,0.0336,17571.0
6239,LTU,2008.0,40.0,100.0,4868.45070,0.0336,17571.0
6240,LTU,2008.0,41.0,100.0,4882.14065,0.0336,17571.0
6241,LTU,2008.0,42.0,100.0,4895.83060,0.0336,17571.0
6242,LTU,2008.0,43.0,100.0,4950.63800,0.0336,17571.0
6243,LTU,2008.0,44.0,100.0,5006.78600,0.0336,17571.0
6244,LTU,2008.0,45.0,100.0,5028.54440,0.0336,17571.0


In [34]:
df_income.rename(columns={'country':'country code', 'year_survey':'year'}, inplace=True)

# Indice de GINI

In [84]:
df_gini.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271 entries, 0 to 270
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Series Name    268 non-null    object 
 1   Series Code    266 non-null    object 
 2   Country Name   266 non-null    object 
 3   Country Code   266 non-null    object 
 4   2004 [YR2004]  76 non-null     float64
 5   2006 [YR2006]  82 non-null     float64
 6   2007 [YR2007]  72 non-null     float64
 7   2008 [YR2008]  79 non-null     float64
 8   2009 [YR2009]  83 non-null     float64
 9   2010 [YR2010]  86 non-null     float64
 10  2011 [YR2011]  82 non-null     float64
dtypes: float64(7), object(4)
memory usage: 23.4+ KB


In [85]:
gini=df_gini.copy()

In [86]:
gini.rename(columns={'Country Code':'country code', 'Country Name':'country'}, inplace=True)

In [87]:
gini.drop(columns=['Series Name','Series Code',], inplace=True)

In [88]:
gini.rename(columns={'2004 [YR2004]':'2004', '2006 [YR2006]':'2006', '2007 [YR2007]':'2007', '2008 [YR2008]':'2008', '2009 [YR2009]':'2009', '2010 [YR2010]':'2010', '2011 [YR2011]':'2011'}, inplace=True)

In [40]:
#df_gini.iloc[:,4:] = df_gini.iloc[:,4:]/100
#df_gini.head(3)

In [89]:
df_gini2 = gini.melt(
    id_vars=["country", "country code"],     # colonnes à conserver
    var_name="year",                         # nom de la nouvelle colonne contenant les années
    value_name="gini"                        # nom de la colonne des valeurs GINI
)

In [90]:
df_gini2["year"] = pd.to_datetime(df_gini2["year"], format="%Y", errors="coerce").dt.year

In [91]:
df_gini2


,country,country code,year,gini
0,World,WLD,2004,NaN
1,Afghanistan,AFG,2004,NaN
2,Albania,ALB,2004,NaN
3,Algeria,DZA,2004,NaN
4,American Samoa,ASM,2004,NaN
...,...,...,...,...
1892,NaN,NaN,2011,NaN
1893,NaN,NaN,2011,NaN
1894,NaN,NaN,2011,NaN
1895,NaN,NaN,2011,NaN


In [44]:
#df_gini2 = df_gini2.dropna(subset=["gini"])


In [92]:
df_gini2.loc[:, "gini"] = df_gini2["gini"] / 100


In [93]:
df_gini2

,country,country code,year,gini
0,World,WLD,2004,NaN
1,Afghanistan,AFG,2004,NaN
2,Albania,ALB,2004,NaN
3,Algeria,DZA,2004,NaN
4,American Samoa,ASM,2004,NaN
...,...,...,...,...
1892,NaN,NaN,2011,NaN
1893,NaN,NaN,2011,NaN
1894,NaN,NaN,2011,NaN
1895,NaN,NaN,2011,NaN


In [94]:
df_main = pd.merge(df_income, df_gini2, how='left', on=['country code', 'year'])
df_main

,country code,year,quantile,nb_quantiles,income,pop,gdpppp,country,gini
0,ALB,2008.0,1.0,100.0,728.89795,0.03143,7297.0,Albania,0.30
1,ALB,2008.0,2.0,100.0,916.66235,0.03143,7297.0,Albania,0.30
2,ALB,2008.0,3.0,100.0,1010.91600,0.03143,7297.0,Albania,0.30
3,ALB,2008.0,4.0,100.0,1086.90780,0.03143,7297.0,Albania,0.30
4,ALB,2008.0,5.0,100.0,1132.69970,0.03143,7297.0,Albania,0.30
...,...,...,...,...,...,...,...,...,...
11595,ZAF,2008.0,96.0,100.0,24553.56800,0.48793,9602.0,South Africa,0.63
11596,ZAF,2008.0,97.0,100.0,28858.03100,0.48793,9602.0,South Africa,0.63
11597,ZAF,2008.0,98.0,100.0,35750.29000,0.48793,9602.0,South Africa,0.63
11598,ZAF,2008.0,99.0,100.0,46297.31600,0.48793,9602.0,South Africa,0.63


In [95]:
df_main.to_csv("/content/drive/MyDrive/Projet_7/df_main.csv", index=False)


In [96]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11600 entries, 0 to 11599
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country code  11600 non-null  object 
 1   year          11600 non-null  float64
 2   quantile      11600 non-null  float64
 3   nb_quantiles  11600 non-null  float64
 4   income        11600 non-null  float64
 5   pop           11600 non-null  float64
 6   gdpppp        11600 non-null  float64
 7   country       11500 non-null  object 
 8   gini          9600 non-null   float64
dtypes: float64(7), object(2)
memory usage: 815.8+ KB


In [97]:
missing_gini = df_main[df_main['gini'].isna()][['country code', 'country', 'year']]
missing_country = list(missing_gini['country code'].unique())
print(f"Nombre de pays sans Gini : {len(missing_country)}")
missing_gini.head()


Nombre de pays sans Gini : 20


,country code,country,year
400,AZE,Azerbaijan,2008.0
401,AZE,Azerbaijan,2008.0
402,AZE,Azerbaijan,2008.0
403,AZE,Azerbaijan,2008.0
404,AZE,Azerbaijan,2008.0


In [98]:
print(missing_country)

['AZE', 'CHN', 'COD', 'EGY', 'GHA', 'GTM', 'IND', 'IRN', 'IRQ', 'KEN', 'KHM', 'LKA', 'MAR', 'MLI', 'MYS', 'PAK', 'SYR', 'TWN', 'XKX', 'YEM']


In [99]:
missing_gini_2008 = df_main[(df_main["gini"].isna()) & (df_main["year"] == 2008)]
liste_pays_manquants_2008 = missing_gini_2008[["country code", "country"]].drop_duplicates().sort_values(by="country")
print(liste_pays_manquants_2008)

      country code             country
400            AZE          Azerbaijan
5700           KHM            Cambodia
2000           COD    Congo, Dem. Rep.
4700           IRN  Iran, Islamic Rep.
4800           IRQ                Iraq
11300          XKX              Kosovo
8300           PAK            Pakistan
11400          YEM         Yemen, Rep.
10500          TWN                 NaN


In [100]:
list_gini = []

# Parcours des couples (pays, année) manquants
for _, row in missing_gini.iterrows():
    code = row['country code']
    year = row['year']

    dep = df_main[(df_main['country code'] == code) & (df_main['year'] == year)]['income'].values

    if len(dep) > 0:
        dep_sorted = np.sort(dep)
        lorenz = np.cumsum(dep_sorted) / dep_sorted.sum()
        n = len(dep)
        AUC = (lorenz.sum() - lorenz[-1]/2 - lorenz[0]/2) / n
        gini = round(2 * (0.5 - AUC), 2)
        list_gini.append((code, year, gini))

# Mise à jour dans le DataFrame principal
for code, year, gini_val in list_gini:
    df_main.loc[(df_main["country code"] == code) & (df_main["year"] == year), "gini"] = gini_val


In [101]:
missing_gini = df_main[df_main['gini'].isna()][['country code', 'country', 'year']]
missing_country = list(missing_gini['country code'].unique())
print(f"Nombre de pays sans Gini : {len(missing_country)}")
missing_gini.head()

Nombre de pays sans Gini : 0


,country code,country,year


In [102]:
print(df_main.columns.tolist())


['country code', 'year', 'quantile', 'nb_quantiles', 'income', 'pop', 'gdpppp', 'country', 'gini']


In [103]:
df_main.columns = df_main.columns.str.strip()  # Pour supprimer les espaces


In [106]:
display(df_main.head())
display(df_main.tail())
display(df_main.shape)

,country code,year,quantile,nb_quantiles,income,pop,gdpppp,country,gini
0,ALB,2008.0,1.0,100.0,728.89795,0.03143,7297.0,Albania,0.3
1,ALB,2008.0,2.0,100.0,916.66235,0.03143,7297.0,Albania,0.3
2,ALB,2008.0,3.0,100.0,1010.91600,0.03143,7297.0,Albania,0.3
3,ALB,2008.0,4.0,100.0,1086.90780,0.03143,7297.0,Albania,0.3
4,ALB,2008.0,5.0,100.0,1132.69970,0.03143,7297.0,Albania,0.3


,country code,year,quantile,nb_quantiles,income,pop,gdpppp,country,gini
11595,ZAF,2008.0,96.0,100.0,24553.568,0.48793,9602.0,South Africa,0.63
11596,ZAF,2008.0,97.0,100.0,28858.031,0.48793,9602.0,South Africa,0.63
11597,ZAF,2008.0,98.0,100.0,35750.290,0.48793,9602.0,South Africa,0.63
11598,ZAF,2008.0,99.0,100.0,46297.316,0.48793,9602.0,South Africa,0.63
11599,ZAF,2008.0,100.0,100.0,82408.550,0.48793,9602.0,South Africa,0.63


(11600, 9)

In [107]:
df_main.dtypes

,0
country code,object
year,float64
quantile,float64
nb_quantiles,float64
income,float64
pop,float64
gdpppp,float64
country,object
gini,float64


In [105]:
display(df_main.isna().any())
display(df_main.duplicated().sum())

,0
country code,False
year,False
quantile,False
nb_quantiles,False
income,False
pop,False
gdpppp,False
country,True
gini,False


np.int64(0)

In [111]:
df_main_pays = df_income[['country code']].drop_duplicates()
df_main_pays.shape

(116, 1)

In [118]:
df_min = pd.merge(df_main, df_main_pays, on="country code", how="inner")
df_min.sample(2)

,country code,year,quantile,nb_quantiles,income,pop,gdpppp,country,gini
6256,LTU,2008.0,57.0,100.0,6047.41060,0.033600,17571.0,Lithuania,0.357
10707,UGA,2009.0,8.0,100.0,290.31378,0.316569,1067.0,Uganda,0.442


In [126]:
df_min.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11600 entries, 0 to 11599
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country code  11600 non-null  object 
 1   year          11600 non-null  float64
 2   quantile      11600 non-null  float64
 3   nb_quantiles  11600 non-null  float64
 4   income        11600 non-null  float64
 5   pop           11600 non-null  float64
 6   gdpppp        11600 non-null  float64
 7   country       11500 non-null  object 
 8   gini          11600 non-null  float64
dtypes: float64(7), object(2)
memory usage: 815.8+ KB


In [125]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11600 entries, 0 to 11599
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country code  11600 non-null  object 
 1   year          11600 non-null  float64
 2   quantile      11600 non-null  float64
 3   nb_quantiles  11600 non-null  float64
 4   income        11600 non-null  float64
 5   pop           11600 non-null  float64
 6   gdpppp        11600 non-null  float64
 7   country       11500 non-null  object 
 8   gini          11600 non-null  float64
dtypes: float64(7), object(2)
memory usage: 815.8+ KB


In [124]:
display(df_min.isna().any())
display(df_min.duplicated().sum())

,0
country code,False
year,False
quantile,False
nb_quantiles,False
income,False
pop,False
gdpppp,False
country,True
gini,False


np.int64(0)

# Partie 2: Analyse exploratoire des inégalités (Mission 2)

* Montrer la diversité des distributions de revenu à l’aide de graphiques log-log

* Représenter les courbes de Lorenz pour chaque pays choisi

* Visualiser l'évolution temporelle de l'indice de Gini

* Résumer les valeurs extrêmes (pays les plus égalitaires et inégalitaires)

In [104]:
# Sélection manuelle de pays avec profils variés
liste_pays = ['FRA', 'USA', 'BRA', 'IND', 'ZAF', 'CHN', 'NOR', 'NGA', 'SWE', 'RUS']

# Filtrer le dataset
df_plot = df_main[df_main["country"].isin(liste_pays)]

# Vérifier les quantiles disponibles
df_plot.groupby("country")["quantile"].nunique()

,quantile
country,
